In [101]:
from pathlib import Path
import shutil
import soundfile as sf


# ==========================================================
# CONFIGURATION
# ==========================================================

# Folder containing:
#
#   Turchet_baseline/
#   test_plugin_turchet/
#   test_yt_sounds/
#
SOURCE_ROOT = Path(
    "static/stimulis"
)


# Two mirrored output datasets.
FIRST_HALF_ROOT = Path(
    "static/stimulis/stimulis_first_half"
)

SECOND_HALF_ROOT = Path(
    "static/stimulis/stimulis_second_half"
)


CONDITIONS = [
    "Turchet_baseline",
    "test_plugin_turchet",
    "test_yt_sounds",
]


HALF_DURATION_S = 5.0


# ==========================================================
# HELPERS
# ==========================================================

def output_name(
    source_path: Path,
    suffix: str,
) -> str:
    """
    Example:

        sound.wav
        ->
        sound_f.wav

    or:

        sound_b.wav
    """

    return (
        source_path.stem
        + suffix
        + source_path.suffix
    )


def split_wav(
    source_path: Path,
    first_output_path: Path,
    second_output_path: Path,
):
    """
    Split a WAV into:

        first half  = 0 -> 5 s
        second half = 5 -> 10 s

    No resampling is performed.
    Channel count is preserved.
    """

    audio, sr = sf.read(
        source_path,
        always_2d=False,
    )


    samples_per_half = int(
        HALF_DURATION_S * sr
    )


    # 0 -> 5 seconds
    first_half = audio[
        :samples_per_half
    ]


    # 5 -> 10 seconds
    second_half = audio[
        samples_per_half:
        2 * samples_per_half
    ]


    first_output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    second_output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    sf.write(
        first_output_path,
        first_half,
        sr,
    )


    sf.write(
        second_output_path,
        second_half,
        sr,
    )


# ==========================================================
# PROCESS DATASET
# ==========================================================

processed = 0
skipped_second_half = 0

# ==========================================================
# CLEAN PREVIOUS SPLIT DATASET
# ==========================================================
#
# Important:
# the splitting code writes new files but does not otherwise
# remove obsolete files from previous runs.
#
# We therefore rebuild both split datasets from scratch.
# ==========================================================

for output_root in [
    FIRST_HALF_ROOT,
    SECOND_HALF_ROOT,
]:

    if output_root.exists():

        print(
            f"Removing old split dataset: "
            f"{output_root}"
        )

        shutil.rmtree(
            output_root
        )


    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )


for condition in CONDITIONS:

    condition_root = (
        SOURCE_ROOT / condition
    )


    if not condition_root.exists():

        print(
            f"WARNING: missing folder: "
            f"{condition_root}"
        )

        continue


    # rglob is deliberate:
    #
    # It finds both:
    #
    #   condition/*.wav
    #
    # and:
    #
    #   condition/originals/*.wav
    #
    wav_files = sorted(
        condition_root.rglob("*.wav")
    )


    print(
        f"\n{condition}: "
        f"{len(wav_files)} WAV files"
    )


    for source_path in wav_files:

        # Preserve the path relative to the condition.
        #
        # Examples:
        #
        #   edit.wav
        #
        #   originals/original.wav
        #
        relative_path = (
            source_path.relative_to(
                condition_root
            )
        )


        relative_parent = (
            relative_path.parent
        )


        first_output_path = (
            FIRST_HALF_ROOT
            / condition
            / relative_parent
            / output_name(
                source_path,
                "_f",
            )
        )


        second_output_path = (
            SECOND_HALF_ROOT
            / condition
            / relative_parent
            / output_name(
                source_path,
                "_b",
            )
        )


        # Read here so we can detect files <= 5 s.
        audio, sr = sf.read(
            source_path,
            always_2d=False,
        )


        split_sample = int(
            HALF_DURATION_S * sr
        )


        first_half = audio[
            :split_sample
        ]


        second_half = audio[
            split_sample:
            2 * split_sample
        ]


        first_output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        second_output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        sf.write(
            first_output_path,
            first_half,
            sr,
        )


        if len(second_half) > 0:

            sf.write(
                second_output_path,
                second_half,
                sr,
            )

        else:

            skipped_second_half += 1

            print(
                "  No second half: "
                f"{source_path.name}"
            )


        processed += 1


print("\n" + "=" * 60)

print(
    f"Processed: {processed} WAV files"
)

print(
    "Files without audio after 5 s: "
    f"{skipped_second_half}"
)

print(
    f"\nFirst halves:\n"
    f"{FIRST_HALF_ROOT}"
)

print(
    f"\nSecond halves:\n"
    f"{SECOND_HALF_ROOT}"
)

Removing old split dataset: static/stimulis/stimulis_first_half
Removing old split dataset: static/stimulis/stimulis_second_half

Turchet_baseline: 396 WAV files

test_plugin_turchet: 372 WAV files

test_yt_sounds: 372 WAV files

Processed: 1140 WAV files
Files without audio after 5 s: 0

First halves:
static/stimulis/stimulis_first_half

Second halves:
static/stimulis/stimulis_second_half


In [1]:
from pathlib import Path

import numpy as np
import soundfile as sf


OUTPUT = Path(
    "static/stimulis/calibration/pink_noise.wav"
)

SR = 48_000
DURATION_S = 6.0

# Moderate target RMS.
# This is intentionally not a claim about physical SPL.
TARGET_RMS = 0.08

rng = np.random.default_rng(42)

n_samples = int(
    SR * DURATION_S
)

# ----------------------------------------------------------
# Generate pink-ish noise in the frequency domain
# ----------------------------------------------------------

freqs = np.fft.rfftfreq(
    n_samples,
    d=1 / SR
)

spectrum = (
    rng.normal(size=len(freqs)) +
    1j * rng.normal(size=len(freqs))
)

# Pink noise has power proportional to 1/f,
# hence amplitude proportional to 1/sqrt(f).
scale = np.zeros_like(freqs)

scale[1:] = (
    1.0 /
    np.sqrt(freqs[1:])
)

spectrum *= scale

noise = np.fft.irfft(
    spectrum,
    n=n_samples
)

# ----------------------------------------------------------
# Normalize RMS
# ----------------------------------------------------------

noise = noise.astype(
    np.float32
)

noise -= noise.mean()

rms = np.sqrt(
    np.mean(noise ** 2)
)

noise *= (
    TARGET_RMS /
    max(rms, 1e-12)
)

# Safety against clipping.
peak = np.max(
    np.abs(noise)
)

if peak > 0.95:
    noise *= 0.95 / peak


# ----------------------------------------------------------
# Short fade-in / fade-out
# ----------------------------------------------------------

fade_samples = int(
    0.05 * SR
)

fade = np.linspace(
    0.0,
    1.0,
    fade_samples,
    dtype=np.float32
)

noise[:fade_samples] *= fade

noise[-fade_samples:] *= fade[::-1]


# Same signal in left and right channels.
stereo = np.column_stack([
    noise,
    noise,
])


OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

sf.write(
    OUTPUT,
    stereo,
    SR,
)

print(
    f"Saved: {OUTPUT}"
)

Saved: static/stimulis/calibration/pink_noise.wav


In [102]:
from pathlib import Path
from math import ceil
import random
import re
from difflib import SequenceMatcher

import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

STIMULI_ROOT = Path("static/stimulis")

HALF_FOLDERS = {
    "first": STIMULI_ROOT / "stimulis_first_half",
    "second": STIMULI_ROOT / "stimulis_second_half",
}


CONDITION_CONFIG = {

    "Turchet_baseline": {
        "nb_var_per_ref_per_participant": 2,
    },

    "test_plugin_turchet": {
        "nb_var_per_ref_per_participant": 3,
    },

    "test_yt_sounds": {
        "nb_var_per_ref_per_participant": 3,
    },
}


N_PARTICIPANTS = 1024

N_REFERENCES_EXPECTED = 12

RANDOM_SEED = 42

In [103]:
# ============================================================
# FILE DISCOVERY
# ============================================================

def get_originals(condition_folder):
    """
    Return the 12 original WAV files in deterministic order.
    """

    originals_folder = (
        condition_folder / "originals"
    )

    originals = sorted(
        originals_folder.glob("*.wav"),
        key=lambda p: p.name.lower(),
    )

    if len(originals) != N_REFERENCES_EXPECTED:
        raise RuntimeError(
            f"{condition_folder}: found "
            f"{len(originals)} originals; "
            f"expected {N_REFERENCES_EXPECTED}."
        )

    return originals


def get_edits(condition_folder):
    """
    Return every WAV directly inside the condition folder.

    WAV files inside originals/ are therefore excluded.
    """

    edits = sorted(
        condition_folder.glob("*.wav"),
        key=lambda p: p.name.lower(),
    )

    return edits

In [104]:
# ============================================================
# ORIGINAL <-> EDIT MATCHING
# ============================================================

def clean_filename_for_matching(path):
    """
    Remove parts of the filename that are known not to
    represent the source identity.
    """

    name = path.stem.lower()

    # Remove half suffix.
    name = re.sub(
        r"_(f|b)$",
        "",
        name,
    )

    # Remove everything beginning with __edit_...
    name = re.sub(
        r"__edit_.*$",
        "",
        name,
    )

    return name


def turchet_baseline_identity(path):
    """
    For Turchet_baseline, the identity of a sound is given
    by the first two underscore-separated words.

    Example
    -------
    dress_gravel_neutral_0010__H2H_1036.82ms__PK_89.32dB_f.wav

        -> dress_gravel
    """

    name = path.stem.lower()

    # Remove _f / _b first.
    name = re.sub(
        r"_(f|b)$",
        "",
        name,
    )

    parts = name.split("_")

    if len(parts) < 2:
        raise ValueError(
            f"Cannot extract Turchet identity from: {path.name}"
        )

    return "_".join(
        parts[:2]
    )


def filename_similarity(
    original_path,
    edited_path,
):
    """
    Generic filename similarity used for datasets for which
    we do not yet have a stronger deterministic rule.
    """

    original_name = (
        clean_filename_for_matching(
            original_path
        )
    )

    edited_name = (
        clean_filename_for_matching(
            edited_path
        )
    )

    return SequenceMatcher(
        None,
        original_name,
        edited_name,
    ).ratio()


def assign_edits_to_originals(
    originals,
    edits,
    condition_name,
):
    """
    Assign edited WAV files to their corresponding originals.

    Matching strategy
    -----------------
    Turchet_baseline:
        Each original is identified by the first two
        underscore-separated words, e.g. "dress_gravel".

        Multiple originals MAY have the same identity.

        In that case, they deliberately share the same set
        of edited sounds.

        Example:

            dress_gravel_neutral_0010_f.wav
            dress_gravel_neutral_0021_f.wav

        both point to all 64 edited sounds whose identity is:

            dress_gravel

    Other conditions:
        For now, assign each edit to the original with the
        highest filename similarity.
    """

    matches = {
        ref_id: []
        for ref_id in range(len(originals))
    }


    # ========================================================
    # TURCHET BASELINE
    # ========================================================

    if condition_name == "Turchet_baseline":

        # ----------------------------------------------------
        # Group edited sounds by their first-two-word identity
        #
        # Example:
        #
        # {
        #     "dress_gravel": [64 paths],
        #     "boots_hay":    [64 paths],
        #     ...
        # }
        # ----------------------------------------------------

        edits_by_identity = {}


        for edit_path in edits:

            identity = (
                turchet_baseline_identity(
                    edit_path
                )
            )


            edits_by_identity.setdefault(
                identity,
                []
            ).append(
                edit_path
            )


        # ----------------------------------------------------
        # Assign edits independently to EACH original.
        #
        # Multiple originals can therefore receive the same
        # list of 64 edited sounds.
        # ----------------------------------------------------

        for ref_id, original_path in enumerate(
            originals
        ):

            identity = (
                turchet_baseline_identity(
                    original_path
                )
            )


            matching_edits = (
                edits_by_identity.get(
                    identity,
                    []
                )
            )


            matches[
                ref_id
            ] = sorted(
                matching_edits,
                key=lambda p: p.name.lower(),
            )


        return matches


    # ========================================================
    # OTHER DATASETS
    # ========================================================

    for edit_path in edits:

        scores = [
            filename_similarity(
                original_path,
                edit_path,
            )
            for original_path in originals
        ]


        best_ref_id = max(
            range(len(scores)),
            key=lambda i: scores[i],
        )


        matches[
            best_ref_id
        ].append(
            edit_path
        )


    return matches

In [105]:
# ============================================================
# INSPECT MATCHING
# ============================================================

matching_summary = []


for half_name, half_root in HALF_FOLDERS.items():

    for condition_name in CONDITION_CONFIG:

        condition_folder = (
            half_root / condition_name
        )


        originals = get_originals(
            condition_folder
        )

        edits = get_edits(
            condition_folder
        )


        matches = assign_edits_to_originals(
            originals,
            edits,
            condition_name,
        )


        for ref_id, original_path in enumerate(originals):

            matching_summary.append({

                "half":
                    half_name,

                "condition":
                    condition_name,

                "ref_id":
                    ref_id,

                "original":
                    original_path.name,

                "n_edits":
                    len(matches[ref_id]),
            })


matching_df = pd.DataFrame(
    matching_summary
)


display(matching_df)

,half,condition,ref_id,original,n_edits
0,first,Turchet_baseline,0,dress_gravel_neutral_0010_f.wav,64
1,first,Turchet_baseline,1,dress_gravel_neutral_0021_f.wav,64
2,first,Turchet_baseline,2,dress_metal_neutral_0241_f.wav,64
3,first,Turchet_baseline,3,dress_metal_neutral_0245_f.wav,64
4,first,Turchet_baseline,4,dress_wood_neutral_0071_f.wav,64
...,...,...,...,...,...
67,second,test_yt_sounds,7,magnus_01_target_b.wav,30
68,second,test_yt_sounds,8,muir_01_target_b.wav,30
69,second,test_yt_sounds,9,sweden_01_target_b.wav,30
70,second,test_yt_sounds,10,sweden_02_target_b.wav,30


In [106]:
# ============================================================
# CREATE INDEXED EDIT METADATA
# ============================================================

def index_edits(matches):
    """
    Give every edit within each reference a stable edit_id.

    No WAV files are renamed.
    """

    indexed = {}


    for ref_id, edit_paths in matches.items():

        sorted_edits = sorted(
            edit_paths,
            key=lambda p: p.name.lower(),
        )


        indexed[ref_id] = [

            {
                "edit_id":
                    edit_id,

                "path":
                    path,

                "filename":
                    path.name,
            }

            for edit_id, path
            in enumerate(sorted_edits)
        ]


    return indexed

In [107]:
# ============================================================
# GENERATE PERMUTATION POOL FOR ONE REFERENCE
# ============================================================

def generate_reference_permutations(
    indexed_edits,
    nb_var_per_ref_per_participant,
    n_participants,
    rng,
):
    """
    Generate enough complete random permutations of all
    available edits to cover at least n_participants.

    Example
    -------
    30 edits
    3 edits / participant

        10 participants per permutation
        ceil(1000 / 10) = 100 permutations

        => 3000 edit IDs


    64 edits
    2 edits / participant

        32 participants per permutation
        ceil(1000 / 32) = 32 permutations

        => 2048 edit IDs
    """

    n_variations = len(
        indexed_edits
    )


    if n_variations == 0:
        raise ValueError(
            "Reference has no edited sounds."
        )


    if (
        n_variations %
        nb_var_per_ref_per_participant
        != 0
    ):

        raise ValueError(
            f"{n_variations} variations cannot be "
            f"evenly divided into groups of "
            f"{nb_var_per_ref_per_participant}."
        )


    participants_per_permutation = (
        n_variations
        //
        nb_var_per_ref_per_participant
    )


    n_permutations = ceil(
        n_participants
        /
        participants_per_permutation
    )


    # Mapping:
    #
    # edit_id -> actual WAV
    edit_by_id = {
        item["edit_id"]: item
        for item in indexed_edits
    }


    permutations = []


    for permutation_id in range(
        n_permutations
    ):

        ids = list(
            range(n_variations)
        )


        rng.shuffle(ids)


        permutation = [

            {
                "permutation_id":
                    permutation_id,

                "edit_id":
                    edit_id,

                "filename":
                    edit_by_id[
                        edit_id
                    ]["filename"],

                "path":
                    str(
                        edit_by_id[
                            edit_id
                        ]["path"]
                    ),
            }

            for edit_id in ids
        ]


        permutations.append(
            permutation
        )


    return permutations

In [108]:
# ============================================================
# GENERATE ALL PERMUTATION POOLS
# ============================================================

rng = random.Random(
    RANDOM_SEED
)


all_permutation_pools = {}


for half_name, half_root in HALF_FOLDERS.items():

    all_permutation_pools[
        half_name
    ] = {}

    for (
        condition_name,
        condition_config
    ) in CONDITION_CONFIG.items():


        print(
            f"\n{'=' * 70}\n"
            f"{half_name} / {condition_name}"
        )

        # ... rest of Cell 7 unchanged


        condition_folder = (
            half_root
            / condition_name
        )


        # ----------------------------------------------------
        # Load the 12 reference/original sounds
        # ----------------------------------------------------

        originals = get_originals(
            condition_folder
        )


        # ----------------------------------------------------
        # Load all edited sounds
        # ----------------------------------------------------

        edits = get_edits(
            condition_folder
        )


        # ----------------------------------------------------
        # Match edits to references
        #
        # IMPORTANT:
        # condition_name is now passed because the matching
        # strategy depends on the dataset.
        #
        # Turchet_baseline:
        #   first two filename components identify the group.
        #   Multiple originals may share the same 64 edits.
        #
        # Other datasets:
        #   use the other matching strategy.
        # ----------------------------------------------------

        matches = assign_edits_to_originals(
            originals,
            edits,
            condition_name,
        )


        # ----------------------------------------------------
        # Give each edit a stable integer ID within its
        # reference.
        # ----------------------------------------------------

        indexed_matches = index_edits(
            matches
        )


        reference_pools = []


        # ----------------------------------------------------
        # Generate permutations independently for each of
        # the 12 reference sounds.
        # ----------------------------------------------------

        for ref_id, original_path in enumerate(
            originals
        ):

            indexed_edits = (
                indexed_matches[
                    ref_id
                ]
            )


            n_variations = len(
                indexed_edits
            )


            # Safety check.
            if n_variations == 0:

                raise RuntimeError(
                    f"No edited sounds found for:\n"
                    f"  half      = {half_name}\n"
                    f"  condition = {condition_name}\n"
                    f"  ref_id    = {ref_id}\n"
                    f"  original  = {original_path.name}"
                )


            # ------------------------------------------------
            # Generate enough full permutations to cover
            # N_PARTICIPANTS.
            # ------------------------------------------------

            permutations = (
                generate_reference_permutations(

                    indexed_edits,

                    nb_var_per_ref_per_participant=
                        condition_config[
                            "nb_var_per_ref_per_participant"
                        ],

                    n_participants=
                        N_PARTICIPANTS,

                    rng=
                        rng,
                )
            )


            # ------------------------------------------------
            # Store everything needed to recover the mapping
            # later.
            # ------------------------------------------------

            reference_pools.append({

                "ref_id":
                    ref_id,

                "original_filename":
                    original_path.name,

                "original_path":
                    str(original_path),

                "n_variations":
                    n_variations,

                "nb_var_per_ref_per_participant":
                    condition_config[
                        "nb_var_per_ref_per_participant"
                    ],

                "n_permutations":
                    len(permutations),

                "permutations":
                    permutations,
            })


            print(
                f"ref {ref_id:02d} | "
                f"{original_path.name} | "
                f"{n_variations} edits | "
                f"{len(permutations)} permutations | "
                f"{len(permutations) * n_variations} "
                f"ordered edit positions"
            )


        # ----------------------------------------------------
        # Store all 12 references for this half / condition
        # ----------------------------------------------------

        all_permutation_pools[
            half_name
        ][
            condition_name
        ] = reference_pools


first / Turchet_baseline
ref 00 | dress_gravel_neutral_0010_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 01 | dress_gravel_neutral_0021_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 02 | dress_metal_neutral_0241_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 03 | dress_metal_neutral_0245_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 04 | dress_wood_neutral_0071_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 05 | dress_wood_neutral_0079_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 06 | sneakers_gravel_neutral_0018_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 07 | sneakers_gravel_neutral_0031_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 08 | sneakers_metal_neutral_0232_f.wav | 64 edits | 32 permutations | 2048 ordered edit positions
ref 09 | sneakers_metal_neutral_0244_f.wav | 64 edits | 32 permutations 

In [109]:
# ============================================================
# CELL 8 — INSPECT GENERATED PERMUTATION POOLS
# ============================================================

import pandas as pd


def inspect_permutation_pool(
    half_name,
    condition_name,
    ref_id=None,
    n_permutations_to_show=3,
    n_items_to_show=15,
):
    """
    Inspect generated permutation pools.

    If ref_id is None:
        Show a summary of all 12 references.

    If ref_id is provided:
        Show detailed information and the first few
        permutations for that reference.
    """

    reference_pools = (
        all_permutation_pools[
            half_name
        ][
            condition_name
        ]
    )


    # ========================================================
    # SUMMARY MODE
    # ========================================================

    if ref_id is None:

        rows = []


        for ref in reference_pools:

            n_positions = (
                ref["n_permutations"]
                *
                ref["n_variations"]
            )

            n_per_participant = (
                ref[
                    "nb_var_per_ref_per_participant"
                ]
            )

            n_participants_covered = (
                n_positions
                //
                n_per_participant
            )


            rows.append({

                "ref_id":
                    ref["ref_id"],

                "original":
                    ref["original_filename"],

                "n_variations":
                    ref["n_variations"],

                "variations_per_participant":
                    n_per_participant,

                "n_permutations":
                    ref["n_permutations"],

                "total_positions":
                    n_positions,

                "participants_covered":
                    n_participants_covered,
            })


        df = pd.DataFrame(rows)

        display(df)

        return df


    # ========================================================
    # DETAILED MODE
    # ========================================================

    ref = reference_pools[
        ref_id
    ]


    print("=" * 80)

    print(
        f"HALF:       {half_name}"
    )

    print(
        f"CONDITION:  {condition_name}"
    )

    print(
        f"REF ID:     {ref['ref_id']}"
    )

    print(
        f"ORIGINAL:   {ref['original_filename']}"
    )

    print(
        f"N EDITS:    {ref['n_variations']}"
    )

    print(
        f"EDITS / PARTICIPANT: "
        f"{ref['nb_var_per_ref_per_participant']}"
    )

    print(
        f"N PERMUTATIONS: "
        f"{ref['n_permutations']}"
    )

    print("=" * 80)


    # ========================================================
    # SHOW INDIVIDUAL PERMUTATIONS
    # ========================================================

    for permutation in ref[
        "permutations"
    ][:n_permutations_to_show]:

        if len(permutation) == 0:
            continue


        permutation_id = (
            permutation[0][
                "permutation_id"
            ]
        )


        edit_ids = [
            item["edit_id"]
            for item in permutation
        ]


        print(
            f"\nPERMUTATION {permutation_id}"
        )

        print(
            "Edit IDs:"
        )

        print(
            edit_ids
        )


        # --------------------------------------------
        # Actual filenames for first N positions
        # --------------------------------------------

        rows = []


        for position, item in enumerate(
            permutation[
                :n_items_to_show
            ]
        ):

            rows.append({

                "position":
                    position,

                "edit_id":
                    item["edit_id"],

                "filename":
                    item["filename"],
            })


        display(
            pd.DataFrame(rows)
        )


    return ref

In [110]:
inspect_permutation_pool(
    "first",
    "Turchet_baseline",
)

,ref_id,original,n_variations,variations_per_participant,n_permutations,total_positions,participants_covered
0,0,dress_gravel_neutral_0010_f.wav,64,2,32,2048,1024
1,1,dress_gravel_neutral_0021_f.wav,64,2,32,2048,1024
2,2,dress_metal_neutral_0241_f.wav,64,2,32,2048,1024
3,3,dress_metal_neutral_0245_f.wav,64,2,32,2048,1024
4,4,dress_wood_neutral_0071_f.wav,64,2,32,2048,1024
5,5,dress_wood_neutral_0079_f.wav,64,2,32,2048,1024
6,6,sneakers_gravel_neutral_0018_f.wav,64,2,32,2048,1024
7,7,sneakers_gravel_neutral_0031_f.wav,64,2,32,2048,1024
8,8,sneakers_metal_neutral_0232_f.wav,64,2,32,2048,1024
9,9,sneakers_metal_neutral_0244_f.wav,64,2,32,2048,1024


,ref_id,original,n_variations,variations_per_participant,n_permutations,total_positions,participants_covered
0,0,dress_gravel_neutral_0010_f.wav,64,2,32,2048,1024
1,1,dress_gravel_neutral_0021_f.wav,64,2,32,2048,1024
2,2,dress_metal_neutral_0241_f.wav,64,2,32,2048,1024
3,3,dress_metal_neutral_0245_f.wav,64,2,32,2048,1024
4,4,dress_wood_neutral_0071_f.wav,64,2,32,2048,1024
5,5,dress_wood_neutral_0079_f.wav,64,2,32,2048,1024
6,6,sneakers_gravel_neutral_0018_f.wav,64,2,32,2048,1024
7,7,sneakers_gravel_neutral_0031_f.wav,64,2,32,2048,1024
8,8,sneakers_metal_neutral_0232_f.wav,64,2,32,2048,1024
9,9,sneakers_metal_neutral_0244_f.wav,64,2,32,2048,1024


In [111]:
inspect_permutation_pool(
    "first",
    "Turchet_baseline",
    ref_id=3,
)

HALF:       first
CONDITION:  Turchet_baseline
REF ID:     3
ORIGINAL:   dress_metal_neutral_0245_f.wav
N EDITS:    64
EDITS / PARTICIPANT: 2
N PERMUTATIONS: 32

PERMUTATION 0
Edit IDs:
[5, 3, 46, 63, 47, 39, 57, 6, 32, 18, 12, 7, 48, 52, 24, 15, 62, 26, 23, 16, 21, 0, 41, 4, 45, 13, 54, 9, 31, 55, 25, 14, 51, 58, 19, 43, 59, 27, 36, 50, 38, 37, 11, 49, 44, 60, 28, 35, 17, 34, 29, 30, 20, 22, 42, 2, 1, 10, 53, 33, 61, 8, 40, 56]


,position,edit_id,filename
0,0,5,dress_metal_neutral_0241__H2H_1137.10ms__PK_64...
1,1,3,dress_metal_neutral_0241__H2H_1034.05ms__PK_85...
2,2,46,dress_metal_neutral_0245__H2H_518.76ms__PK_77....
3,3,63,dress_metal_neutral_0245__H2H_930.99ms__PK_85....
4,4,47,dress_metal_neutral_0245__H2H_518.76ms__PK_85....
5,5,39,dress_metal_neutral_0245__H2H_1137.10ms__PK_85...
6,6,57,dress_metal_neutral_0245__H2H_827.93ms__PK_64....
7,7,6,dress_metal_neutral_0241__H2H_1137.10ms__PK_72...
8,8,32,dress_metal_neutral_0245__H2H_1034.05ms__PK_56...
9,9,18,dress_metal_neutral_0241__H2H_621.82ms__PK_77....



PERMUTATION 1
Edit IDs:
[51, 38, 31, 37, 28, 58, 23, 13, 20, 49, 44, 32, 60, 2, 21, 9, 3, 48, 54, 43, 1, 36, 0, 61, 6, 55, 30, 22, 46, 50, 19, 29, 7, 53, 8, 11, 4, 12, 62, 42, 34, 18, 17, 39, 33, 57, 40, 59, 5, 16, 14, 52, 63, 41, 47, 45, 10, 15, 25, 27, 35, 56, 26, 24]


,position,edit_id,filename
0,0,51,dress_metal_neutral_0245__H2H_621.82ms__PK_81....
1,1,38,dress_metal_neutral_0245__H2H_1137.10ms__PK_77...
2,2,31,dress_metal_neutral_0241__H2H_930.99ms__PK_81....
3,3,37,dress_metal_neutral_0245__H2H_1137.10ms__PK_68...
4,4,28,dress_metal_neutral_0241__H2H_930.99ms__PK_56....
5,5,58,dress_metal_neutral_0245__H2H_827.93ms__PK_72....
6,6,23,dress_metal_neutral_0241__H2H_724.87ms__PK_81....
7,7,13,dress_metal_neutral_0241__H2H_518.76ms__PK_64....
8,8,20,dress_metal_neutral_0241__H2H_724.87ms__PK_56....
9,9,49,dress_metal_neutral_0245__H2H_621.82ms__PK_64....



PERMUTATION 2
Edit IDs:
[54, 38, 40, 18, 35, 33, 31, 14, 37, 45, 22, 19, 26, 41, 59, 32, 27, 46, 8, 52, 43, 51, 1, 62, 20, 56, 10, 55, 57, 47, 36, 13, 6, 61, 60, 58, 39, 28, 42, 15, 7, 0, 24, 16, 12, 11, 30, 34, 4, 25, 49, 17, 3, 9, 48, 53, 5, 21, 2, 50, 44, 29, 23, 63]


,position,edit_id,filename
0,0,54,dress_metal_neutral_0245__H2H_724.87ms__PK_77....
1,1,38,dress_metal_neutral_0245__H2H_1137.10ms__PK_77...
2,2,40,dress_metal_neutral_0245__H2H_1240.16ms__PK_56...
3,3,18,dress_metal_neutral_0241__H2H_621.82ms__PK_77....
4,4,35,dress_metal_neutral_0245__H2H_1034.05ms__PK_81...
5,5,33,dress_metal_neutral_0245__H2H_1034.05ms__PK_64...
6,6,31,dress_metal_neutral_0241__H2H_930.99ms__PK_81....
7,7,14,dress_metal_neutral_0241__H2H_518.76ms__PK_72....
8,8,37,dress_metal_neutral_0245__H2H_1137.10ms__PK_68...
9,9,45,dress_metal_neutral_0245__H2H_518.76ms__PK_68....


{'ref_id': 3,
 'original_filename': 'dress_metal_neutral_0245_f.wav',
 'original_path': 'static/stimulis/stimulis_first_half/Turchet_baseline/originals/dress_metal_neutral_0245_f.wav',
 'n_variations': 64,
 'nb_var_per_ref_per_participant': 2,
 'n_permutations': 32,
 'permutations': [[{'permutation_id': 0,
    'edit_id': 5,
    'filename': 'dress_metal_neutral_0241__H2H_1137.10ms__PK_64.58dB_f.wav',
    'path': 'static/stimulis/stimulis_first_half/Turchet_baseline/dress_metal_neutral_0241__H2H_1137.10ms__PK_64.58dB_f.wav'},
   {'permutation_id': 0,
    'edit_id': 3,
    'filename': 'dress_metal_neutral_0241__H2H_1034.05ms__PK_85.61dB_f.wav',
    'path': 'static/stimulis/stimulis_first_half/Turchet_baseline/dress_metal_neutral_0241__H2H_1034.05ms__PK_85.61dB_f.wav'},
   {'permutation_id': 0,
    'edit_id': 46,
    'filename': 'dress_metal_neutral_0245__H2H_518.76ms__PK_77.20dB_f.wav',
    'path': 'static/stimulis/stimulis_first_half/Turchet_baseline/dress_metal_neutral_0245__H2H_518.76m

In [112]:
for half_name in ["first", "second"]:

    print(f"\n{half_name}:")

    for condition_name, pools in (
        all_permutation_pools[
            half_name
        ].items()
    ):

        print(
            f"  {condition_name:25s}"
            f" -> {len(pools)} references"
        )


first:
  Turchet_baseline          -> 12 references
  test_plugin_turchet       -> 12 references
  test_yt_sounds            -> 12 references

second:
  Turchet_baseline          -> 12 references
  test_plugin_turchet       -> 12 references
  test_yt_sounds            -> 12 references


In [113]:
# ============================================================
# CELL 10 — GET EDITS FOR ONE PARTICIPANT / REFERENCE
# ============================================================


def flatten_reference_pool(
    reference_pool,
):
    """
    Concatenate all permutations for one reference.

    Returns the edit records in presentation-pool order.
    """

    flattened = []

    for permutation in reference_pool[
        "permutations"
    ]:

        flattened.extend(
            permutation
        )

    return flattened


def get_participant_edits(
    reference_pool,
    participant_id,
):
    """
    Return the edits assigned to one participant for one
    reference.

    participant_id is 1-based.

    Example with 3 edits/participant:

        participant 1 -> positions 0:3
        participant 2 -> positions 3:6
        participant 3 -> positions 6:9

    Example with 2 edits/participant:

        participant 1 -> positions 0:2
        participant 2 -> positions 2:4
        participant 3 -> positions 4:6
    """

    if participant_id < 1:
        raise ValueError(
            "participant_id must start at 1."
        )


    n_per_participant = (
        reference_pool[
            "nb_var_per_ref_per_participant"
        ]
    )


    flattened = flatten_reference_pool(
        reference_pool
    )


    start = (
        participant_id - 1
    ) * n_per_participant

    stop = (
        start
        + n_per_participant
    )


    if stop > len(flattened):

        raise RuntimeError(
            f"Participant {participant_id} "
            f"cannot be assigned.\n"
            f"Need positions {start}:{stop}, "
            f"but pool contains only "
            f"{len(flattened)} positions."
        )


    return flattened[
        start:stop
    ]

In [114]:
# ============================================================
# CELL 11 — BUILD 192 TRIALS FOR ONE PARTICIPANT
# ============================================================


def build_participant_trials(
    participant_id,
    all_permutation_pools,
    seed=42,
):
    """
    Build the complete 192-trial Phase 2 plan for one
    participant.

    Exactly half the trials have:

        A = original
        B = edited

    and half:

        A = edited
        B = original

    All 192 trials are finally shuffled together.
    """

    rng = random.Random(
        f"{seed}_participant_{participant_id}"
    )


    trials = []


    # ========================================================
    # COLLECT ALL ASSIGNED EDITS
    # ========================================================

    for half_name in [
        "first",
        "second",
    ]:

        for condition_name in [
            "Turchet_baseline",
            "test_plugin_turchet",
            "test_yt_sounds",
        ]:

            reference_pools = (
                all_permutation_pools[
                    half_name
                ][
                    condition_name
                ]
            )


            if len(reference_pools) != 12:

                raise RuntimeError(
                    f"{half_name} / "
                    f"{condition_name}: "
                    f"expected 12 references, "
                    f"found {len(reference_pools)}."
                )


            for reference_pool in reference_pools:

                ref_id = (
                    reference_pool[
                        "ref_id"
                    ]
                )

                original_path = (
                    reference_pool[
                        "original_path"
                    ]
                )

                original_filename = (
                    reference_pool[
                        "original_filename"
                    ]
                )


                participant_edits = (
                    get_participant_edits(
                        reference_pool,
                        participant_id,
                    )
                )


                for edit in participant_edits:

                    trials.append({

                        "participant_id":
                            participant_id,

                        "half":
                            half_name,

                        "condition":
                            condition_name,

                        "ref_id":
                            ref_id,

                        "original_filename":
                            original_filename,

                        "original_path":
                            original_path,

                        "edit_id":
                            edit["edit_id"],

                        "edited_filename":
                            edit["filename"],

                        "edited_path":
                            edit["path"],
                    })


    # ========================================================
    # CHECK TOTAL
    # ========================================================

    if len(trials) != 192:

        raise RuntimeError(
            f"Participant {participant_id}: "
            f"expected 192 trials, "
            f"generated {len(trials)}."
        )


    # ========================================================
    # BALANCE A/B DIRECTION EXACTLY 50/50
    # ========================================================
    #
    # 96:
    #   A = original
    #
    # 96:
    #   A = edited
    #
    # Shuffle the orientations first so there is no pattern.
    # ========================================================

    orientations = (
        ["original_A"] * 96
        +
        ["edited_A"] * 96
    )

    rng.shuffle(
        orientations
    )


    for trial, orientation in zip(
        trials,
        orientations,
    ):

        if orientation == "original_A":

            trial["sound_A_type"] = (
                "original"
            )

            trial["sound_A_path"] = (
                trial["original_path"]
            )

            trial["sound_B_type"] = (
                "edited"
            )

            trial["sound_B_path"] = (
                trial["edited_path"]
            )

        else:

            trial["sound_A_type"] = (
                "edited"
            )

            trial["sound_A_path"] = (
                trial["edited_path"]
            )

            trial["sound_B_type"] = (
                "original"
            )

            trial["sound_B_path"] = (
                trial["original_path"]
            )


    # ========================================================
    # MIX EVERYTHING
    # ========================================================

    rng.shuffle(
        trials
    )


    # ========================================================
    # FINAL TRIAL INDEX
    # ========================================================

    for trial_index, trial in enumerate(
        trials
    ):

        trial[
            "trial_index"
        ] = trial_index


    return trials

In [115]:
participant_1 = build_participant_trials(
    participant_id=1,
    all_permutation_pools=all_permutation_pools,
)

len(participant_1)

192

In [116]:
# ============================================================
# CELL 12 — INSPECT ONE PARTICIPANT
# ============================================================

participant_1_df = pd.DataFrame(
    participant_1
)


display(
    participant_1_df[
        [
            "trial_index",
            "half",
            "condition",
            "ref_id",
            "edit_id",
            "sound_A_type",
            "sound_A_path",
            "sound_B_type",
            "sound_B_path",
        ]
    ]
)

,trial_index,half,condition,ref_id,edit_id,sound_A_type,sound_A_path,sound_B_type,sound_B_path
0,0,first,test_yt_sounds,5,10,original,static/stimulis/stimulis_first_half/test_yt_so...,edited,static/stimulis/stimulis_first_half/test_yt_so...
1,1,first,test_plugin_turchet,0,12,edited,static/stimulis/stimulis_first_half/test_plugi...,original,static/stimulis/stimulis_first_half/test_plugi...
2,2,second,test_yt_sounds,10,9,edited,static/stimulis/stimulis_second_half/test_yt_s...,original,static/stimulis/stimulis_second_half/test_yt_s...
3,3,first,Turchet_baseline,9,10,edited,static/stimulis/stimulis_first_half/Turchet_ba...,original,static/stimulis/stimulis_first_half/Turchet_ba...
4,4,second,test_plugin_turchet,1,11,edited,static/stimulis/stimulis_second_half/test_plug...,original,static/stimulis/stimulis_second_half/test_plug...
...,...,...,...,...,...,...,...,...,...
187,187,second,test_plugin_turchet,5,23,edited,static/stimulis/stimulis_second_half/test_plug...,original,static/stimulis/stimulis_second_half/test_plug...
188,188,first,test_plugin_turchet,8,2,original,static/stimulis/stimulis_first_half/test_plugi...,edited,static/stimulis/stimulis_first_half/test_plugi...
189,189,second,Turchet_baseline,11,8,original,static/stimulis/stimulis_second_half/Turchet_b...,edited,static/stimulis/stimulis_second_half/Turchet_b...
190,190,second,test_yt_sounds,11,7,edited,static/stimulis/stimulis_second_half/test_yt_s...,original,static/stimulis/stimulis_second_half/test_yt_s...


In [117]:
print(
    "Total trials:",
    len(participant_1_df)
)


print("\nTrials per condition / half:")

display(
    participant_1_df.groupby(
        [
            "half",
            "condition",
        ]
    ).size()
)


print("\nA/B orientation:")

display(
    participant_1_df[
        "sound_A_type"
    ].value_counts()
)

Total trials: 192

Trials per condition / half:


half    condition          
first   Turchet_baseline       24
        test_plugin_turchet    36
        test_yt_sounds         36
second  Turchet_baseline       24
        test_plugin_turchet    36
        test_yt_sounds         36
dtype: int64


A/B orientation:


sound_A_type
original    96
edited      96
Name: count, dtype: int64

In [118]:
# ============================================================
# CELL 14 — GENERATE AND SAVE ALL PARTICIPANT TRIAL ORDERS
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_PATH = Path(
    "static/js/phase2_trial_orders.json"
)

N_FINAL_PARTICIPANTS = 1024


# ------------------------------------------------------------
# GENERATE ALL PARTICIPANTS
# ------------------------------------------------------------

participant_trial_orders = {}


for participant_id in range(
    1,
    N_FINAL_PARTICIPANTS + 1,
):

    trials = build_participant_trials(
        participant_id=participant_id,
        all_permutation_pools=all_permutation_pools,
    )


    # Final safety check
    if len(trials) != 192:

        raise RuntimeError(
            f"Participant {participant_id}: "
            f"expected 192 trials, "
            f"found {len(trials)}."
        )


    participant_trial_orders[
        str(participant_id)
    ] = trials


# ------------------------------------------------------------
# SAVE JSON
# ------------------------------------------------------------

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        participant_trial_orders,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

file_size_mb = (
    OUTPUT_PATH.stat().st_size
    / 1024
    / 1024
)


print(
    f"Saved {N_FINAL_PARTICIPANTS} participants."
)

print(
    f"Trials per participant: 192"
)

print(
    f"Total trials: "
    f"{N_FINAL_PARTICIPANTS * 192:,}"
)

print(
    f"File: {OUTPUT_PATH.resolve()}"
)

print(
    f"Size: {file_size_mb:.2f} MB"
)

Saved 1024 participants.
Trials per participant: 192
Total trials: 196,608
File: /Users/etiennebost/Desktop/code/AudioConceptSlidersExperiment2026/static/js/phase2_trial_orders.json
Size: 165.43 MB


In [119]:
# ============================================================
# CELL 15 — VALIDATE SAVED FILE
# ============================================================

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8",
) as f:

    saved_orders = json.load(f)


print(
    "Participants:",
    len(saved_orders)
)


# Inspect participant 1
p1 = saved_orders["1"]

print(
    "Participant 1 trials:",
    len(p1)
)


# Check A/B balance
n_original_A = sum(
    trial["sound_A_type"] == "original"
    for trial in p1
)

n_edited_A = sum(
    trial["sound_A_type"] == "edited"
    for trial in p1
)


print(
    "Original as A:",
    n_original_A
)

print(
    "Edited as A:",
    n_edited_A
)


# Display first 10 trials
display(
    pd.DataFrame(
        p1[:10]
    )[
        [
            "trial_index",
            "half",
            "condition",
            "ref_id",
            "edit_id",
            "sound_A_type",
            "sound_A_path",
            "sound_B_type",
            "sound_B_path",
        ]
    ]
)

Participants: 1024
Participant 1 trials: 192
Original as A: 96
Edited as A: 96


,trial_index,half,condition,ref_id,edit_id,sound_A_type,sound_A_path,sound_B_type,sound_B_path
0,0,first,test_yt_sounds,5,10,original,static/stimulis/stimulis_first_half/test_yt_so...,edited,static/stimulis/stimulis_first_half/test_yt_so...
1,1,first,test_plugin_turchet,0,12,edited,static/stimulis/stimulis_first_half/test_plugi...,original,static/stimulis/stimulis_first_half/test_plugi...
2,2,second,test_yt_sounds,10,9,edited,static/stimulis/stimulis_second_half/test_yt_s...,original,static/stimulis/stimulis_second_half/test_yt_s...
3,3,first,Turchet_baseline,9,10,edited,static/stimulis/stimulis_first_half/Turchet_ba...,original,static/stimulis/stimulis_first_half/Turchet_ba...
4,4,second,test_plugin_turchet,1,11,edited,static/stimulis/stimulis_second_half/test_plug...,original,static/stimulis/stimulis_second_half/test_plug...
5,5,first,test_plugin_turchet,5,20,original,static/stimulis/stimulis_first_half/test_plugi...,edited,static/stimulis/stimulis_first_half/test_plugi...
6,6,first,test_yt_sounds,9,29,edited,static/stimulis/stimulis_first_half/test_yt_so...,original,static/stimulis/stimulis_first_half/test_yt_so...
7,7,first,test_plugin_turchet,4,12,original,static/stimulis/stimulis_first_half/test_plugi...,edited,static/stimulis/stimulis_first_half/test_plugi...
8,8,first,test_plugin_turchet,5,5,edited,static/stimulis/stimulis_first_half/test_plugi...,original,static/stimulis/stimulis_first_half/test_plugi...
9,9,second,test_plugin_turchet,3,12,original,static/stimulis/stimulis_second_half/test_plug...,edited,static/stimulis/stimulis_second_half/test_plug...


In [120]:
# ============================================================
# DIAGNOSTIC — YOUTUBE FILENAMES
# ============================================================

for half_name, half_root in HALF_FOLDERS.items():

    folder = (
        half_root
        / "test_yt_sounds"
    )

    originals = get_originals(
        folder
    )

    edits = get_edits(
        folder
    )

    print("\n" + "=" * 80)
    print(
        f"{half_name.upper()} HALF"
    )

    print(
        f"\nOriginals: {len(originals)}"
    )

    for path in originals:
        print(
            "  ORIGINAL:",
            path.name
        )

    print(
        f"\nEdits: {len(edits)}"
    )

    for path in edits[:40]:
        print(
            "  EDIT:",
            path.name
        )


FIRST HALF

Originals: 12
  ORIGINAL: catacombs_01_target_f.wav
  ORIGINAL: forest_01_target_f.wav
  ORIGINAL: forest_02_target_f.wav
  ORIGINAL: forest_03_target_f.wav
  ORIGINAL: free_hike_01_target_f.wav
  ORIGINAL: free_hike_02_target_f.wav
  ORIGINAL: hiking_01_target_f.wav
  ORIGINAL: magnus_01_target_f.wav
  ORIGINAL: muir_01_target_f.wav
  ORIGINAL: sweden_01_target_f.wav
  ORIGINAL: sweden_02_target_f.wav
  ORIGINAL: vatican_03_target_f.wav

Edits: 360
  EDIT: catacombs_01_target__edit_00__alpha_m1p766776_f.wav
  EDIT: catacombs_01_target__edit_01__alpha_p8p101689_f.wav
  EDIT: catacombs_01_target__edit_02__alpha_m0p164920_f.wav
  EDIT: catacombs_01_target__edit_03__alpha_p2p589969_f.wav
  EDIT: catacombs_01_target__edit_04__alpha_m9p492819_f.wav
  EDIT: catacombs_01_target__edit_05__alpha_p9p768575_f.wav
  EDIT: catacombs_01_target__edit_06__alpha_m0p711915_f.wav
  EDIT: catacombs_01_target__edit_07__alpha_m7p060815_f.wav
  EDIT: catacombs_01_target__edit_08__alpha_p20p07488